In [3]:
# Bibliotecas e Configurações
import sys
from pathlib import Path
import json

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision import transforms
from torchinfo import summary

# Visualização e métricas
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import PIL

# Adiciona a raiz do projeto ao path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from config import (
    get_dataset_paths
)

from src.dataset import DataModuleParasite

In [22]:
# Configuration for Egg Dataset
CONFIG_EGG = {
    'dataset_name': 'eggs',
    'split': 1,
    'percentage': [5, 25, 50, 75, 100],
    'num_classes': 9,
    'batch_size': 16,
    'num_workers': 4,
    'image_size': 200,
    'lr': 1e-4,
    'num_epochs': 25,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# Create dataloaders
dataloaders_perc = {}

transforms_egg = transforms.Compose([
    transforms.Resize((CONFIG_EGG['image_size'], CONFIG_EGG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

for perc in CONFIG_EGG['percentage']:
    data_module = DataModuleParasite(
        dataset_name=CONFIG_EGG['dataset_name'],
        split=CONFIG_EGG['split'],
        percentage=perc,
        batch_size=CONFIG_EGG['batch_size'],
        num_workers=CONFIG_EGG['num_workers'],
        transform=transforms_egg
    )
    data_module.setup()
    dataloaders_perc[perc] = {
        'train': data_module.train_dataloader(),
        'val': data_module.val_dataloader(),
        'test': data_module.test_dataloader()
    }

In [ ]:
# Functions for training and evaluation

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def train(model, train_loader, criterion, optimizer, device):
    pass

def evaluate():
    pass

In [ ]:
# Model
vgg16_model_pretrained = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
vgg16_model_pretrained.classifier[6] = nn.Linear(in_features=4096, out_features=CONFIG_EGG['num_classes'])
vgg16_model_pretrained = vgg16_model_pretrained.to(CONFIG_EGG['device'])

total_params = sum(p.numel() for p in vgg16_model_pretrained.parameters())
trainable_params = sum(p.numel() for p in vgg16_model_pretrained.parameters() if p.requires_grad)

# print(f"Total parameters: {total_params}")
# print(f"Trainable parameters: {trainable_params}")
# print(summary(vgg16_model_pretrained, input_size=(1, 3, 224, 224)))

# Loss , Optimizer and Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16_model_pretrained.parameters(), lr=CONFIG_EGG['lr'])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# Training loop
history_perc = {} 
best_val_acc = 0.0
best_val_kap = 0.0

for perc in CONFIG_EGG['percentage']:
    best_val_acc = 0.0
    best_val_kap = 0.0
    
    train_loss, train_acc, train_kap = train(vgg16_model_pretrained, dataloaders_perc[perc]['train'], criterion, optimizer, CONFIG_EGG['device'])

    # val_loss, val_acc, val_kap = evaluate(vgg16_model_pretrained, dataloaders_perc[perc]['val'], criterion, CONFIG_EGG['device'])

    scheduler.step()
    
    history_perc[perc] = {
        'train_loss': train_loss,
        'train_acc': train_acc,
        'train_kap': train_kap,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_kap': val_kap
    }
    
    # Save the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(vgg16_model_pretrained.state_dict(), f'vgg16_egg_best_acc_perc_{perc}.pth')
